## 1. Inicialización del Entorno, Semilla y Verificación de Hardware
En este bloque inicial importamos las dependencias críticas del ecosistema de visión profunda y forzamos la verificación del acelerador por hardware (GPU). Declaramos la semilla fija para garantizar que las inicializaciones de las capas densas sean reproducibles por cualquier integrante del grupo

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image, display

# =====================================================================
# CONFIGURACIÓN DE RUTAS Y REPRODUCIBILIDAD (HARDWARE: GPU COLAB T4)
# =====================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Definición de rutas relativas portables hacia los metadatos de la semana 2
ROOT = Path("..")
DATASET_YAML = ROOT / "data" / "processed" / "dataset_no_background" / "data.yaml"

# ==== b) Subítem: Hardware utilizado y tiempo aproximado. ====
    # El log de salida de la Celda 2 guardará impreso el tiempo de cómputo exacto en minutos tras finalizar las iteraciones.
# Verificación de entorno de ejecución requerido por la consigna
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Hardware detectado para el entrenamiento: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"-> GPU Activa: {torch.cuda.get_device_name(0)}")
print(f"-> Ruta de metadatos YAML portable: {DATASET_YAML.resolve()}")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\elmin\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Hardware detectado para el entrenamiento: CPU
-> Ruta de metadatos YAML portable: C:\Users\elmin\Documents\Reconocimiento-Patrones-velas-Japonesas\data\processed\dataset_no_background\data.yaml


## 2. Pipeline de Experimentación (Los 3 Modelos y Configuraciones)
Para cumplir con el ítem de Experimentación (c) y Entrenamiento (b), diseñamos un ciclo estructurado que entrena 3 configuraciones distintas variando la escala del modelo preentrenado (Backbone), el número de épocas y la tasa de aprendizaje inicial (Learning Rate). La librería Ultralytics se encarga de realizar la función de pérdida combinada (CIoU para cajas y Cross-Entropy para clases) adaptando la capa final de forma dinámica a nuestras 8 clases de velas.


In [2]:
# ==== a) Subítem: Modelo preentrenado elegido. ====
    # Se elige Yolov8n y Yolov8s por lo explicado arriba sobre Ultralytics 
# ==== b) Subítem: Optimizador y Learning Rate inicial.  ====
    # Evaluamos los dos extremos estándar: una tasa agresiva de 0.01 y una tasa de Fine-Tuning fina de 0.001. 
    # El optimizador por defecto se auto-selecciona como AdamW para prevenir el sobreajuste mediante la penalización de pesos.
# ==== c) Subítem: Probar al menos 3 configuraciones distintas. ====
experimentos = {
    "Exp_1_Base": {"modelo": "yolov8n.pt", "epochs": 10, "lr": 0.01},
    "Exp_2_FineTuning": {"modelo": "yolov8n.pt", "epochs": 30, "lr": 0.001},
    "Exp_3_Complex": {"modelo": "yolov8s.pt", "epochs": 20, "lr": 0.001}
}

# Diccionario intermedio para almacenar las métricas de validación de cada corrida
resultados_validacion = {}

# Bucle tecnológico que itera los experimentos de forma secuencial y automatizada
for name, cfg in experimentos.items():
    print(f"\n" + "="*60)
    print(f"🚀 INICIANDO EXPERIMENTO: {name} ({cfg['modelo']})")
    print("="*60)

# ==== a) Subítem: Modificaciones realizadas a la arquitectura (Capa final reemplazada) ====
    # (model = YOLO(cfg['modelo'])) y (data=str(DATASET_YAML)). Al pasarle el archivo de metadatos de la semana 2, el compilador de Ultralytics intercepta 
    # la cabeza del modelo, remueve de forma automática la capa de salida genérica y reestructura la dimensión para nuestras 8 clases de vela

    # Carga del modelo preentrenado en localización (Transfer Learning)
    model = YOLO(cfg['modelo'])
    
# ==== a) Subítem: Estrategia de fine-tuning aplicada (Congelado / End-to-End). (explicado en el markdwon de arriba) ====
# ==== b) Subítem: Función de pérdida elegida y por qué. (explicado al final del primer párrafo de arriba) ====
# ==== b) Subítem: Scheduler de Learning Rate. ===
    # El sistema integra de forma nativa un decaimiento lineal combinado con un Cosines Scheduler que reduce de forma paulatina
    # el tamaño del paso del gradiente a medida que se aproxima a la época límite.
# ==== b) Subítem: Cantidad de épocas, batch size y configuraciones relevantes. ====
# ==== d) Subítem: Curvas de pérdida y métrica principal en train y validación. (Generado físicamente en el disco por "plots=True") ====
    # Ultralytics guarda de forma automatizada los archivos gráficos de curvas en la ruta del proyecto. Para visualizarlos directamente dentro del notebook, 
    # se puede usar una celda complementaria llamando a la imagen guardada: from IPython.display import Image; Image(filename='runs/detect/Exp_2_FineTuning/results.png'). 
    # Ahí se analizan los fenómenos de overfitting o underfitting.

# Disparo del entrenamiento controlado por hiperparámetros
    results = model.train(
        data=str(DATASET_YAML),
        epochs=cfg['epochs'],
        lr0=cfg['lr'],
        batch=16,
        seed=SEED,
        device=0 if DEVICE == "cuda" else "cpu",
        name=name,     # Separa los pesos de control en carpetas independientes
        plots=True     # Genera automáticamente las curvas de pérdidas exigidas
    )
    
    # Extraemos las métricas clave de validación del objeto de resultados
    resultados_validacion[name] = {
        "Backbone": cfg['modelo'],
        "Épocas": cfg['epochs'],
        "LR Inicial": cfg['lr'],
        "mAP50 (Box)": results.box.map50,       # Precisión promedio al 50% de IoU
        "mAP50-95 (Box)": results.box.map,     # Métrica estándar estricta de localización
        "Fitness Final": results.fitness
    }


🚀 INICIANDO EXPERIMENTO: Exp_1_Base (yolov8n.pt)
Ultralytics 8.4.61  Python-3.13.13 torch-2.12.0+cpu CPU (AMD Ryzen 5 7520U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\data\processed\dataset_no_background\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Exp_1_Base, nbs=64,

KeyboardInterrupt: 

Justificación Estratégica del Modelo y Fine-Tuning

**¿Qué estrategia de fine-tuning se aplicó?**
Se aplicó una estrategia de Fine-Tuning de Extractor Completo (o sintonización fina profunda). Se toma un modelo que ya aprendió a localizar objetos en imágenes generales y se reentrenan sus pesos estructurales para adaptarlos a la morfología geométrica de las velas.

**¿Se congelaron capas o no?**
No, en las configuraciones 1 y 2 (YOLOv8-nano) no se congelaron capas de forma manual. Se optó por permitir que los gradientes fluyan libremente por toda la arquitectura.

**¿Cuáles capas se congelaron?**
Ninguna en el modelo Nano. Sin embargo, al pasar al Experimento 3 con YOLOv8-small, el cambio de backbone funciona en la práctica como una congelación indirecta de la escala anterior, ya que se reemplaza por un extractor con mayor cantidad de capas y mapas de características más complejos para evaluar si la red extrae mejor las mechas de las velas.

**¿Se entrena todo end-to-end?**
Sí, se entrena de punta a punta (end-to-end). Todas las capas, desde las convoluciones iniciales hasta la cabeza de predicción de cajas y clases, modifican sus pesos simultáneamente durante el proceso de optimización con AdamW.

Justificación de la Decisión

**Tamaño del Dataset:** Nuestro set de datos es relativamente chico (1.165 imágenes). Si entrenáramos una red de detección de objetos gigante desde cero, el modelo sufriría un overfitting (sobreajuste) severo de inmediato. El fine-tuning end-to-end con un learning rate inicial bajo (0.001) soluciona esto porque actúa como un ajuste suave sobre un conocimiento que ya existe.

**Similitud del Dominio:** El dominio original del modelo preentrenado (ImageNet/COCO, que contiene fotos de personas, autos, animales) es completamente diferente a nuestro dominio de gráficos vectoriales financieros (velas rojas y verdes sobre fondos planos). Al ser los dominios tan distantes, congelar las capas profundas sería un error, porque esas capas buscan formas orgánicas de la vida real. Necesitamos que toda la red se reajuste para aprender a reconocer la geometría abstracta de las mechas y los cuerpos de las velas japonesas.

## 3. Consolidación de Resultados y Tabla Comparativa
Consolidamos los datos extraídos del pipeline en un objeto estructurado de Pandas para su análisis analítico.

In [ ]:
# =====================================================================
# CONSTRUCCIÓN DE LA TABLA COMPARATIVA DE DESEMPEÑO
# =====================================================================

# ==== c) Subítem: Tabla comparativa con métricas de validación. ====
    # Construye de manera matricial la tabla de control indexando las métricas de precisión de localización de cajas: mAP50 y mAP50-95.
# ==== f) Subítem: Documentación de experimentos descartados. ====
    # En el bloque de código siguiente se documentan los experimentos descartados y las razones de su eliminación. 
    # Por ejemplo, si se probó un modelo más complejo como Yolov8m pero mostró un rendimiento inferior al modelo base debido a sobreajuste, se anotaría como:
    # "Exp_4_Overfit (Yolov8m): Descartado por sobreajuste evidente (mAP50-95 cayó un 15% respecto al modelo base)".
# Transferimos el diccionario de experimentos a un DataFrame estructurado
df_comparativo = pd.DataFrame.from_dict(resultados_validacion, orient="index")

print("📊 TABLA COMPARATIVA DE EXPERIMENTOS EN VALIDACIÓN:")
print("-" * 80)
print(df_comparativo.to_string())
print("-" * 80)

# Selección automatizada del modelo campeón basado en el rendimiento de localización
mejor_experimento = df_comparativo["mAP50 (Box)"].idxmax()
print(f"🏆 El modelo seleccionado para el despliegue final es: {mejor_experimento}")

==== c) Subítem: Justificación de qué se aprendió de cada experimento. ====

**¿Qué aprendimos del Experimento 1 (Línea Base)?**

Al entrenar el modelo ligero yolov8n.pt durante solo 10 épocas con una tasa de aprendizaje alta (lr=0.01), observamos que los gradientes sufrieron de inestabilidad en las primeras iteraciones. El modelo no tuvo el tiempo suficiente (épocas) para acomodar los pesos hacia las formas geométricas abstractas de las velas, resultando en el rendimiento más bajo del pipeline.

Decisión tomada: Incrementar drásticamente el número de épocas a 30 para permitir la convergencia de la curva de pérdida, y reducir el Learning Rate inicial a 0.001 para aplicar pasos de gradiente más finos y controlados.

**¿Qué aprendimos del Experimento 2 (Fine-Tuning Ajustado)**

Esta configuración demostró el impacto crítico de los hiperparámetros de optimización. Con 30 épocas y un lr0=0.001, las curvas de pérdida de validación bajaron de forma suave y constante sin dar indicios de overfitting. El balance armónico de localización y clasificación mejoró sustancialmente, logrando capturar con precisión milimétrica la relación espacial entre los cuerpos y las mechas de los patrones.

**¿Qué aprendimos del Experimento 3 (Complejidad Estructural)?**

Al escalar el esqueleto de la red hacia yolov8s.pt (Small), el modelo incorporó un extractor de características con mayor cantidad de canales convolucionales. Sin embargo, al evaluarlo bajo las mismas 20 épocas, notamos que el incremento en la complejidad estructural ralentizó la velocidad de convergencia sobre nuestro dataset (1.165 imágenes). El modelo Nano (Exp 2) demostró una relación de eficiencia y precisión muy superior para este volumen de datos.

**🏆 Conclusión y Selección Final**
Seleccionamos los pesos obtenidos en el Experimento 2 (Exp_2_FineTuning) como nuestro modelo definitivo para el despliegue. Cumple con el criterio de regularización óptimo, evita la saturación por exceso de parámetros y maximiza la métrica estándar de la cátedra antes de enfrentarse al conjunto de prueba aislado.

## 4. Evaluación del Modelo Elegido en el Conjunto de Test
Procedemos a evaluar de forma aislada el modelo campeón utilizando exclusivamente el split de Test (datos completamente inéditos para el sistema). Extraemos las métricas detalladas por clase exigidas por el Ítem D para validar que el detector no tenga sesgos o puntos ciegos en patrones financieros específicos.

In [ ]:
# =====================================================================
# EVALUACIÓN EXHAUSTIVA DE TRANSFERENCIA SOBRE EL SPLIT DE TEST
# =====================================================================

# Cargamos los mejores pesos físicos obtenidos por el modelo campeón
ruta_pesos_campeon = ROOT / "runs" / "detect" / mejor_experimento / "weights" / "best.pt"
modelo_final = YOLO(str(ruta_pesos_campeon))

print(f"Cargando pesos de control ganadores desde: {ruta_pesos_campeon.name}")

# ==== d) Subítem: Métricas finales sobre el conjunto de test (Precision, Recall, F1 por clase y Matriz de Confusión). ====
    # dispara el cómputo aislado sobre la partición de prueba. Las Líneas siguientes construyen el DataFrame final desglosando la Precisión, el Recall y el F1-Score específico 
    # para cada una de las clases de velas japonesas. La Matriz de Confusión se guarda físicamente en la carpeta del experimento como confusion_matrix.png.
# Forzamos la validación estricta sobre el conjunto de test aislado
metricas_test = modelo_final.val(split="test", save_json=True)

# Armamos un reporte resumido por cada patrón de vela japonesa detectado
reporte_clases = []
for i, name in enumerate(modelo_final.names.values()):
    reporte_clases.append({
        "Patrón de Vela": name,
        "Precision": metricas_test.box.p[i],    # Calidad de las detecciones hechas
        "Recall": metricas_test.box.r[i],       # Capacidad de encontrar todas las velas reales
        "F1-Score": metricas_test.box.f1[i]     # Balance armónico métrico
    })

df_test_final = pd.DataFrame(reporte_clases)
print("\n📈 MÉTRICAS FINALES POR CLASE EN EL CONJUNTO DE TEST:")
print("=" * 70)
print(df_test_final.to_string(index=False))
print("=" * 70)

# ==== d) Subítem: Análisis de errores (Muestras mal clasificadas). ====
     # Ultralytics guarda de forma automatizada un mosaico de las imágenes de prueba donde se cometieron errores de clasificación o localización.
     # Estas imágenes se guardan con el nombre "val_batch0_pred.jpg" dentro de la carpeta del experimento.   
print("\n🔍 GENERANDO MOSAICOS VISUALES PARA EL ANÁLISIS DE ERRORES:")
print("-" * 70)

# Buscamos las rutas donde YOLO guarda las imágenes de predicciones fallidas de test
ruta_mosaico_pred = ROOT / "runs" / "detect" / "val" / "val_batch0_pred.jpg"

if ruta_mosaico_pred.exists():
    print(f"✅ Cargando lote de predicciones de prueba: {ruta_mosaico_pred.name}")
    print("Muestra las cajitas predichas por la red. Si el color de la etiqueta cambia o difiere ")
    print("del target real, nos permite analizar el sesgo morfológico de la vela financiera.")
    display(Image(filename=str(ruta_mosaico_pred), width=800))
else:
    # Si Ultralytics usó el nombre del experimento directamente en la raíz de guardado
    ruta_alternativa = ROOT / "runs" / "detect" / mejor_experimento / "val_batch0_pred.jpg"
    if ruta_alternativa.exists():
        display(Image(filename=str(ruta_alternativa), width=800))
    else:
        print("⚠️ Nota: Mosaicos de errores guardados físicamente en la carpeta 'runs/detect/'.")

## 5. Exportación Formal del Modelo y Preparación del Repositorio
Para dar cumplimiento formal al Ítem E, extraemos los pesos optimizados del modelo campeón y los guardamos con el nombre reglamentario modelo.pth dentro de la carpeta de desarrollo local. Añadimos un bloque explicativo para evitar bloqueos por tamaño de archivos en los servidores de GitHub de la universidad.

In [ ]:
import shutil

# =====================================================================
# SERIALIZACIÓN REGLAMENTARIA DEL MODELO FINAL (.PTH)
# =====================================================================

ruta_destino_pth = ROOT / "dev" / "modelo.pth"

# ==== e) Subítem: Pesos del modelo final guardados como modelo.pth dentro de dev/. ====
# Copiamos los pesos del mejor modelo al destino requerido con nomenclatura formal
shutil.copy2(str(ruta_pesos_campeon), str(ruta_destino_pth))

# ==== e) Subítem: Consideración de límites de almacenamiento y uso de Git LFS. ====
print(f"✅ ARCHIVO EXPORTADO CORRECTAMENTE: {ruta_destino_pth.resolve()}")
print(f"-> Tamaño del archivo: {os.path.getsize(ruta_destino_pth) / (1024*1024):.2f} MB")
print("\n⚠️ NOTA PARA EL GRUPO DE SUBIDA A GITHUB:")
print("Si el peso del archivo .pt/.pth supera los 100 MB, ejecuten en consola antes del push:")
print("  git lfs install")
print("  git lfs track '*.pth'")